[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR/blob/main/colab_train_MA_A.ipynb)

# Part A: the final runs

Use a GPU runtime (**Runtime → Change runtime type → L4 GPU**). The cells clone the repository, install it, mount Drive, start TensorBoard and then run `train.py`, one call per run. The hyperparameters of every run are in `config.py`:

1. **Pretrain**: the final DAE (salt and pepper 0.5, h 256, dropout 0.4, 20 epochs, the epoch with the best val k-NN is kept) and the AE (the same, `--noise_level 0`).
2. **Frozen evaluation** of the DAE, the AE and a random encoder: k-NN (k=20, cosine), linear probe and analysis (A4 robustness, A5 evidence), on Tiny-ImageNet and CIFAR-100.
3. **Fine-tuning** of the same three encoders on 1%, 10% and 100% of the labels, on both datasets.
4. **Supervised baseline**: the same encoder trained from scratch with labels on 1%, 10% and 100%, and the frozen evaluation of the 100% one.

47 runs, seed 42, about 6 to 8 hours on an L4. Every run writes to `midterm_final` on Drive and is scored on the test set (only val chooses the checkpoint). After a disconnect, run all again: a finished run (it has `results.json`) is skipped, an unfinished one continues from its `last.ckpt`.

## Setup
Downloads the repository (or pulls the latest code), installs it and checks the GPU. Use an L4 runtime.

In [ ]:
import os

REPO = "/content/Representation-Learning-Midterm-DAE-SimCLR"
if not os.path.exists(REPO):
    !git clone https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR.git {REPO}
os.chdir(REPO)
!git pull
!git log --oneline -1

In [ ]:
%pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → L4 GPU, then rerun this cell."
print(torch.cuda.get_device_name(0), "|", os.cpu_count(), "CPU cores")

## Drive
`OUT` is where every run writes.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/לימודים/תואר שני/שנה ב/סמסטר א/למידת ייצוגים במודלים עמוקים/מטלות/מטלת אמצע"
OUT = f"{DRIVE_DIR}/midterm_final"
os.makedirs(OUT, exist_ok=True)
print("writes to:", OUT)

## TensorBoard
Filter the runs with part of a name (`pretrain`, `linear_probe`, `finetune`, `seed1`), then *Toggle all runs*. The `epoch` section has train, val and test on the same axis; the TEXT tab has each run's `run_info`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{OUT}"

## Settings
`BACKBONE` chooses the encoder of every run. The report uses the small encoder of the course (CNN-3); ResNet-18 is an option.

In [ ]:
BACKBONE = "small"       # CNN-3, the course encoder: the runs of the report
# BACKBONE = "resnet18"  # optional: the same runs on ResNet-18 (config.py was tuned for CNN-3)
SEED = 42                # the training seed of every run
COMMON = f'--backbone {BACKBONE} --seed {SEED} --output_dir "{OUT}"'

ENCODERS = ("dae", "dae --noise_level 0", "random")  # DAE, AE, random
DATASETS = ("tinyimagenet", "cifar100")
FRACTIONS = (0.01, 0.1, 1.0)

## 1. Pretrain: the DAE and the AE

In [ ]:
!python train.py --mode pretrain {COMMON}
!python train.py --mode pretrain --noise_level 0 {COMMON}

## 2. Frozen evaluation: k-NN, linear probe, analysis
The linear probe uses the defaults of `config.py`.

In [ ]:
for encoder in ENCODERS:
    for dataset in DATASETS:
        !python train.py --mode knn --encoder {encoder} --dataset {dataset} {COMMON}
        !python train.py --mode linear_probe --encoder {encoder} --dataset {dataset} {COMMON}
        !python train.py --mode analysis --encoder {encoder} --dataset {dataset} {COMMON}

## 3. Fine-tuning on 1%, 10% and 100% of the labels
With the recipe tuned on val (`config.TUNED`); the learning rate of the new head is 3 times that of the encoder.

In [ ]:
for encoder in ENCODERS:
    for dataset in DATASETS:
        for fraction in FRACTIONS:
            !python train.py --mode finetune --encoder {encoder} --dataset {dataset} --label_fraction {fraction} {COMMON}

## 4. The supervised baseline
The same encoder trained from scratch with labels (the same recipe), then the frozen evaluation of the one trained on 100%.

In [ ]:
for fraction in FRACTIONS:
    !python train.py --mode pretrain_supervise_encoder --label_fraction {fraction} {COMMON}
for dataset in DATASETS:
    !python train.py --mode knn --encoder supervised --dataset {dataset} {COMMON}
    !python train.py --mode linear_probe --encoder supervised --dataset {dataset} {COMMON}
    !python train.py --mode analysis --encoder supervised --dataset {dataset} {COMMON}